# Experiment 20: Target + Frequency Encoding + XGBoost

A different modeling strategy from the previous experiments. This experiment tests frequency encoding and leakage-safe out-of-fold target encoding for the categorical variables in the EV dataset.\n

Previous local best: **0.941815**\n

The target encoding for training rows is generated out-of-fold. Validation rows are encoded using statistics learned only from the training split.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path(r'C:\Users\aakif\Documents\DataCompetition')
DATA_PATH = PROJECT_ROOT / 'data' / 'train.csv'

train = pd.read_csv(DATA_PATH)
        
X = train.drop(columns=['Will_Buy_EV', 'id']).copy()
y = train['Will_Buy_EV'].map({'No': 0, 'Yes': 1})

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()

print('Train shape:', X_train.shape)
print('Validation shape:', X_valid.shape)
print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))
print('Categorical columns:', categorical_features)


Train shape: (534932, 13)
Validation shape: (133733, 13)
Numeric features: 7
Categorical features: 6
Categorical columns: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


In [2]:
def clean_categories(df, columns):
    result = df.copy()
    for col in columns:
        result[col] = result[col].astype('string').fillna('__MISSING__')
    return result


def add_frequency_encoding(train_df, valid_df, categorical_columns):
    train_out = train_df.copy()
    valid_out = valid_df.copy()

    for col in categorical_columns:
        train_values = train_out[col].astype('string').fillna('__MISSING__')
        valid_values = valid_out[col].astype('string').fillna('__MISSING__')

        frequencies = train_values.value_counts(normalize=True)

        train_out[f'{col}__freq'] = train_values.map(frequencies).fillna(0.0).astype(float)
        valid_out[f'{col}__freq'] = valid_values.map(frequencies).fillna(0.0).astype(float)

    return train_out, valid_out


def make_target_maps(train_series, target_series, smoothing):
    temp = pd.DataFrame({
        'category': train_series.values,
        'target': target_series.values
    })

    global_mean = float(target_series.mean())
    stats = temp.groupby('category')['target'].agg(['mean', 'count'])

    smoothed = (
        (stats['count'] * stats['mean'] + smoothing * global_mean)
        / (stats['count'] + smoothing)
    )

    return smoothed, global_mean


def add_oof_target_encoding(X_tr, y_tr, X_va, categorical_columns, smoothing):
    X_tr_out = X_tr.copy()
    X_va_out = X_va.copy()
        
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for col in categorical_columns:
        train_values = X_tr_out[col].astype('string').fillna('__MISSING__')
        valid_values = X_va_out[col].astype('string').fillna('__MISSING__')

        oof_values = pd.Series(index=X_tr_out.index, dtype=float)

        for fit_idx, holdout_idx in skf.split(X_tr_out, y_tr):
            fit_index = X_tr_out.index[fit_idx]
            holdout_index = X_tr_out.index[holdout_idx]
        
            mapping, global_mean = make_target_maps(
                train_values.loc[fit_index],
                y_tr.loc[fit_index],
                smoothing
            )
        
            oof_values.loc[holdout_index] = (
                train_values.loc[holdout_index]
                .map(mapping)
                .fillna(global_mean)
                .astype(float)
            )
        
        full_mapping, full_global_mean = make_target_maps(
            train_values,
            y_tr,
            smoothing
        )
        
        validation_values = (
            valid_values
            .map(full_mapping)
            .fillna(full_global_mean)
            .astype(float)
        )
        
        X_tr_out[f'{col}__target'] = oof_values.astype(float)
        X_va_out[f'{col}__target'] = validation_values

    return X_tr_out, X_va_out


In [3]:
def make_model():
    return XGBClassifier(
        n_estimators=800,
        max_depth=5,
        learning_rate=0.04,
        min_child_weight=2,
        subsample=0.90,
        colsample_bytree=0.85,
        gamma=0,
        reg_alpha=0,
        reg_lambda=1,
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )


def prepare_features(X_tr, y_tr, X_va, mode, smoothing):
    tr = clean_categories(X_tr, categorical_features)
    va = clean_categories(X_va, categorical_features)

    if mode in ['frequency', 'frequency_target', 'onehot_frequency_target']:
        tr, va = add_frequency_encoding(tr, va, categorical_features)

    if mode in ['target', 'frequency_target', 'onehot_frequency_target']:
        tr, va = add_oof_target_encoding(
            tr,
            y_tr,
            va,
            categorical_features,
            smoothing
        )

    return tr, va


def fit_and_score(X_tr, y_tr, X_va, y_va, keep_original_categories):
    train_data = X_tr.copy()
    valid_data = X_va.copy()

    if not keep_original_categories:
        train_data = train_data.drop(columns=categorical_features)
        valid_data = valid_data.drop(columns=categorical_features)

    num_cols = train_data.select_dtypes(include=['number']).columns.tolist()
    cat_cols = train_data.select_dtypes(exclude=['number']).columns.tolist()

    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median'))
    ])

    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    transformers = []

    if num_cols:
        transformers.append(('num', numeric_pipeline, num_cols))

    if cat_cols:
        transformers.append(('cat', categorical_pipeline, cat_cols))

    preprocessor = ColumnTransformer(transformers)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', make_model())
    ])

    pipeline.fit(train_data, y_tr)
    predictions = pipeline.predict_proba(valid_data)[:, 1]
    score = roc_auc_score(y_va, predictions)

    return score


In [4]:
results = []

configs = [
    ('20A_Frequency', 'frequency', 0, False),
    ('20B_Target_Smoothing_5', 'target', 5, False),
    ('20C_Target_Smoothing_20', 'target', 20, False),
    ('20D_Target_Smoothing_50', 'target', 50, False),
    ('20E_Frequency_Target_20', 'frequency_target', 20, False),
    ('20F_Frequency_Target_50', 'frequency_target', 50, False),
    ('20G_OneHot_Frequency_Target_20', 'onehot_frequency_target', 20, True),
    ('20H_OneHot_Frequency_Target_50', 'onehot_frequency_target', 50, True)
]

for name, mode, smoothing, keep_original in configs:
    print('\n' + '=' * 70)
    print(name)
    print('=' * 70)

    X_tr_encoded, X_va_encoded = prepare_features(
        X_train,
        y_train,
        X_valid,
        mode,
        smoothing
    )

    print('Encoded feature count:', X_tr_encoded.shape[1])
    print('New numeric encoded columns:', len([
        c for c in X_tr_encoded.columns
        if '__freq' in c or '__target' in c
    ]))

    score = fit_and_score(
        X_tr_encoded,
        y_train,
        X_va_encoded,
        y_valid,
        keep_original
    )

    results.append({
        'Experiment': name,
        'Features': X_tr_encoded.shape[1],
        'ROC_AUC': score
    })

    print(f'ROC-AUC: {score:.6f}')



20A_Frequency
Encoded feature count: 19
New numeric encoded columns: 6
ROC-AUC: 0.941750

20B_Target_Smoothing_5
Encoded feature count: 19
New numeric encoded columns: 6
ROC-AUC: 0.941741

20C_Target_Smoothing_20
Encoded feature count: 19
New numeric encoded columns: 6
ROC-AUC: 0.941741

20D_Target_Smoothing_50
Encoded feature count: 19
New numeric encoded columns: 6
ROC-AUC: 0.941741

20E_Frequency_Target_20
Encoded feature count: 25
New numeric encoded columns: 12
ROC-AUC: 0.941741

20F_Frequency_Target_50
Encoded feature count: 25
New numeric encoded columns: 12
ROC-AUC: 0.941741

20G_OneHot_Frequency_Target_20
Encoded feature count: 25
New numeric encoded columns: 12
ROC-AUC: 0.941738

20H_OneHot_Frequency_Target_50
Encoded feature count: 25
New numeric encoded columns: 12
ROC-AUC: 0.941738


In [5]:
results_df = pd.DataFrame(results).sort_values(
    'ROC_AUC',
    ascending=False
).reset_index(drop=True)

print('\n' + '=' * 70)
print('EXPERIMENT 20 RESULTS')
print('=' * 70)
print(results_df.to_string(index=False))

best_score = float(results_df.loc[0, 'ROC_AUC'])
best_name = results_df.loc[0, 'Experiment']
previous_best = 0.941815
difference = best_score - previous_best

print('\nPrevious local best:', f'{previous_best:.6f}')
print('Best Experiment 20 model:', best_name)
print('Best Experiment 20 ROC-AUC:', f'{best_score:.6f}')
print('Difference vs previous best:', f'{difference:+.6f}')

if best_score > previous_best:
    print('\n🔥 NEW LOCAL BEST')
else:
    print('\nNo Experiment 20 model beat the current local best.')



EXPERIMENT 20 RESULTS
                    Experiment  Features  ROC_AUC
                 20A_Frequency        19 0.941750
       20E_Frequency_Target_20        25 0.941741
       20F_Frequency_Target_50        25 0.941741
        20B_Target_Smoothing_5        19 0.941741
       20D_Target_Smoothing_50        19 0.941741
       20C_Target_Smoothing_20        19 0.941741
20G_OneHot_Frequency_Target_20        25 0.941738
20H_OneHot_Frequency_Target_50        25 0.941738

Previous local best: 0.941815
Best Experiment 20 model: 20A_Frequency
Best Experiment 20 ROC-AUC: 0.941750
Difference vs previous best: -0.000065

No Experiment 20 model beat the current local best.
